In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge, RidgeCV, ElasticNet,ElasticNetCV, LassoCV, lasso_path, enet_path
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline, Pipeline

import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import LeaveOneOut, cross_validate, KFold

from ISLP.models import (ModelSpec as MS, summarize , poly)


# Nuisance Data

In [ ]:
n = 100
p = 100

np.random.seed(318)

x1 = np.random.uniform(0, 1, size=n)
y = 1 + 2 * x1 + np.random.normal(loc=0, scale=0.1, size=n)
x_rest = np.random.normal(loc=0, scale=0.01,size=(n, p-1))

X_full = np.column_stack([x1, x_rest])  # shape: (n, p)
x_cols = ["x1"] + [f"x{i}" for i in range(2, p + 1)]

df = pd.DataFrame(X_full, columns=x_cols)
df["y"] = y

# fit model with all features
X = df.drop(columns='y')
# response
Y = df['y']

# Ridge Regression
`alpha` corresponds to $\lambda$:

In [ ]:
model = Ridge(alpha=1.0)
model.fit(X, Y)
model.intercept_, model.coef_[0:4]

In [ ]:
X.mean()

In [ ]:
X.std()

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled

In [ ]:
X_scaled.mean(0)

In [ ]:
model = Ridge(alpha=1.0)
model.fit(X_scaled, Y)
model.intercept_, model.coef_[0:4]

In [ ]:
Y.mean()

In [ ]:
scaler.mean_

In [ ]:
scaler.scale_

## Systematic Ridge Regression and Cross Validation
To pick the optimal value of the regularization, we need to fit on a variety of $\lambda$'s, and then identify where the MSE (for instance) is minimal, having estimated it by CV methods.

### Manual Scaling

In [ ]:
X_scaled = scaler.fit_transform(X)

In [ ]:
alphas = np.logspace(-2, 3, 50)

model = Ridge()

K = 5  # number of folds for cross-validation
cv5 = KFold(n_splits=K, shuffle=True, random_state=318)

param_grid = {
    'alpha': alphas,
}

grid = GridSearchCV(
    model,
    param_grid,
    scoring='neg_mean_squared_error',
    cv=cv5,
    return_train_score=True
)

grid.fit(X_scaled, Y)

In [ ]:
grid.best_params_, grid.best_estimator_.intercept_, grid.best_estimator_.coef_[0:4]

In [ ]:
fig, ax = plt.subplots()
ax.errorbar(alphas, -grid.cv_results_['mean_test_score'], 
            yerr=grid.cv_results_['std_test_score']/np.sqrt(K))
ax.axvline(grid.best_params_['alpha'], color='red', linestyle='--', label='Best alpha')
ax.set_xscale('log')
ax.set_xlabel(r"$\lambda$")
ax.set_ylabel("CV MSE")
ax.set_title("Ridge CV Curve")

#### Coefficient Paths

In [ ]:
coef_path = []

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_scaled, Y)
    coef_path.append(ridge.coef_)

coef_path = np.array(coef_path)

coefs = pd.DataFrame(coef_path, columns=X.columns, index=alphas)

fig, ax = plt.subplots()
coefs.plot(ax=ax, legend=False)
ax.axvline(grid.best_params_['alpha'], color='red', linestyle='--', label='Best alpha')

ax.set_xscale('log')
ax.set_xlabel(r"$\lambda$")
ax.set_ylabel("Coefficient Value")
ax.set_title("Ridge Coefficient Paths")

### With Pipelines
Pipelines allow you to push data through a sequence of transforms all at once

In [ ]:
alphas = np.logspace(-2, 3, 50)

# use a pipeline to standardize then fit
model = make_pipeline(StandardScaler(), Ridge())
K = 5 # number of folds for cross-validation
cv5 = KFold(n_splits=K, shuffle=True, random_state=318)

param_grid = {
    'ridge__alpha': alphas
}

grid = GridSearchCV(
    model,
    param_grid,
    scoring='neg_mean_squared_error',
    cv=cv5,
    return_train_score=True
)

grid.fit(X, y)

Accessing the components of the pipeline:

In [ ]:
grid.best_estimator_[0], grid.best_estimator_[1]

In [ ]:
grid.best_estimator_[0].mean_[0:4], grid.best_estimator_[0].scale_[0:4]

In [ ]:
grid.best_estimator_.named_steps['standardscaler'], grid.best_estimator_.named_steps['ridge']

In [ ]:
grid.best_params_, grid.best_estimator_.named_steps['ridge'].intercept_, grid.best_estimator_.named_steps['ridge'].coef_[0:4]

In [ ]:
fig, ax = plt.subplots()
ax.errorbar(alphas, -grid.cv_results_['mean_test_score'], 
            yerr=grid.cv_results_['std_test_score']/np.sqrt(K))
ax.axvline(grid.best_params_['ridge__alpha'], color='red', linestyle='--', label='Best alpha')
ax.set_xscale('log')
ax.set_xlabel(r"$\lambda$")
ax.set_ylabel("CV MSE")
ax.set_title("Ridge CV Curve")
# fig.savefig("ridge_cv_curve.pdf")


In [ ]:
ridge = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
ridge.fit(X, y)


In [ ]:
ridge.named_steps['ridge'].coef_

In [ ]:
coef_path = []

for alpha in alphas:
    model_ = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    model_.fit(X, y)
    coef_path.append(model_[1].coef_)

coef_path = np.array(coef_path)

coefs = pd.DataFrame(coef_path, columns=X.columns, index=alphas)

fig, ax = plt.subplots()
coefs.plot(ax=ax, legend=False)
ax.axvline(grid.best_params_['ridge__alpha'], color='red', linestyle='--', label='Best alpha')

ax.set_xscale('log')
ax.set_xlabel(r"$\lambda$")
ax.set_ylabel("Coefficient Value")
ax.set_title("Ridge Coefficient Paths")


In [ ]:
grid.best_estimator_.named_steps['ridge'].intercept_ - np.sum(grid.best_estimator_.named_steps['ridge'].coef_/scaler.scale_ * scaler.mean_)

In [ ]:
grid.best_estimator_.named_steps['ridge'].coef_/scaler.scale_

# LASSO

## Manual Scaling

In [ ]:
X_scaled = scaler.fit_transform(X)

In [ ]:
alphas = np.logspace(-3, 2, 50)

cv5 = KFold(n_splits=5, shuffle=True, random_state=318)
lasso_cv = LassoCV(alphas=alphas, cv=cv5)
lasso_cv.fit(X_scaled, Y)

In [ ]:
lasso_cv.alpha_

In [ ]:
fig, ax = plt.subplots()
ax.errorbar(lasso_cv.alphas_, lasso_cv.mse_path_.mean(1),
            yerr=lasso_cv.mse_path_.std(1)/np.sqrt(K))
ax.axvline(lasso_cv.alpha_, color='red', linestyle='--', label='Best alpha')
ax.set_xscale('log')
ax.set_xlabel(r"$\lambda$")
ax.set_ylabel("CV MSE")
ax.set_title("LASSO CV Curve")
fig.savefig('lasso_cv_cuve.pdf')

In [ ]:
lasso_cv.alpha_, lasso_cv.intercept_, lasso_cv.coef_[0:4]

In [ ]:
np.where(np.abs(lasso_cv.coef_) > 0), lasso_cv.coef_[np.where(np.abs(lasso_cv.coef_) > 0)]

Automatically get the coefficients along the path; this assumes that you've already scaled:

In [ ]:
alphas_path, coefs_, _ = lasso_path(X_scaled, Y, alphas=alphas)
coefs = pd.DataFrame(coefs_.T, columns=X.columns, index=alphas_path)

In [ ]:
fig, ax = plt.subplots()
coefs.plot(ax=ax, legend=False)
ax.axvline(lasso_cv.alpha_, color='red', linestyle='--', label='Best alpha')
ax.set_xscale('log')
ax.set_xlabel(r"$\lambda$")
ax.set_ylabel("Coefficient Value")
ax.set_title("LASSO Coefficient Paths")
fig.savefig('lasso_coefficient_paths.pdf')

In [ ]:
lasso_cv.coef_/scaler.scale_

In [ ]:
lasso_cv.intercept_- np.sum(lasso_cv.coef_/scaler.scale_ * scaler.mean_)

## With Pipelines

In [ ]:
alphas = np.logspace(-3, 2, 50)

cv5 = KFold(n_splits=5, shuffle=True, random_state=318)
model = make_pipeline(StandardScaler(), LassoCV(alphas=alphas, cv=cv5))
# lasso_cv = LassoCV(alphas=alphas, cv=cv5)
model.fit(X, Y)

In [ ]:
model[0].mean_[0:4],model[0].scale_[0:4]

In [ ]:
model[1].alpha_, model[1].intercept_, model[1].coef_[0:4]

In [ ]:
alphas_path, coefs_, _ = lasso_path(X_scaled, Y, alphas=alphas)
coefs = pd.DataFrame(coefs_.T, columns=X.columns, index=alphas_path)

fig, ax = plt.subplots()
coefs.plot(ax=ax, legend=False)
ax.axvline(model[1].alpha_, color='red', linestyle='--', label='Best alpha')
ax.set_xscale('log')
ax.set_xlabel(r"$\lambda$")
ax.set_ylabel("Coefficient Value")
ax.set_title("LASSO Coefficient Paths")

